# Análisis exploratorio · StreamView Analytics

**Evaluación Parcial N°1 · ADY1104 Visualización de Datos**
Escuela de Informática y Telecomunicaciones · DUOC UC, Sede Los Lagos
Equipo consultor: **Felipe Ángel · Daniel Vargas**

---

### Pregunta que guía el cuaderno

> Si el presupuesto de contenido se mantuviera igual, ¿dónde habría que moverlo
> para capturar más atención y mejor satisfacción por cada dólar invertido?

Este cuaderno recorre el camino desde los dos archivos originales hasta los
hallazgos que sostienen el informe ejecutivo. Reutiliza los módulos de `src/`
—no reimplementa ningún cálculo— para que el cuaderno, el informe, el dashboard
y la presentación no puedan dar cifras distintas.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config as cfg      # rutas, paleta y estilo compartidos
import etl                # integración y saneamiento
import metricas as M      # todos los indicadores del proyecto
import graficos as G      # las figuras del informe

import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print("Proyecto:", cfg.RAIZ.name)

Proyecto: EP1_StreamView_Analytics


## 1. Las fuentes en crudo

Antes de integrar nada conviene mirar qué llega. Dos archivos, mismo esquema
salvo la capa financiera, que sólo existe para películas.

In [2]:
peliculas = pd.read_csv(cfg.CSV_PELICULAS)
series = pd.read_csv(cfg.CSV_SERIES)

print(f"Películas: {peliculas.shape[0]:,} filas × {peliculas.shape[1]} columnas")
print(f"Series:    {series.shape[0]:,} filas × {series.shape[1]} columnas")
print("\nSólo en películas:", sorted(set(peliculas.columns) - set(series.columns)))
peliculas.head(3)

Películas: 16,000 filas × 18 columnas
Series:    16,000 filas × 16 columnas

Sólo en películas: ['budget', 'revenue']


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000,752600867
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000,839030630
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000,954305868


## 2. Cuatro trampas en los datos

Las cuatro observaciones siguientes cambian por completo lo que se puede
graficar. Encontrarlas antes de dibujar evita construir un gráfico
conceptualmente incorrecto.

### 2.1 `rating` es una copia exacta de `vote_average`

Dos columnas con la misma información inflan artificialmente el esquema y
pueden llevar a contar dos veces la misma variable en una matriz de
correlación.

In [3]:
for nombre, df in [("películas", peliculas), ("series", series)]:
    print(f"{nombre:<12} coincidencia rating == vote_average: "
          f"{(df.rating == df.vote_average).mean():.1%}")

películas    coincidencia rating == vote_average: 100.0%
series       coincidencia rating == vote_average: 100.0%


### 2.2 La muestra está balanceada por diseño

Exactamente 1.000 títulos por año en cada archivo. **Cualquier gráfico de
volumen en el tiempo sería una línea plana por construcción** y no diría nada
sobre el negocio. Sólo se pueden leer composiciones y tasas dentro de cada
año.

In [4]:
conteo = pd.DataFrame({
    "Películas": peliculas.release_year.value_counts().sort_index(),
    "Series": series.release_year.value_counts().sort_index(),
})
print(conteo.T.to_string())
print("\n¿Todos los años tienen exactamente 1.000 títulos?",
      bool((conteo == 1000).all().all()))

release_year  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
Películas     1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000
Series        1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000

¿Todos los años tienen exactamente 1.000 títulos? True


### 2.3 El 0 en presupuesto y recaudación significa «no informado»

Si se dejara como 0, el promedio de presupuesto caería a menos de un tercio de
su valor real y aparecerían películas «gratis» con recaudaciones millonarias.

In [5]:
print(f"budget  == 0: {(peliculas.budget == 0).mean():.1%}")
print(f"revenue == 0: {(peliculas.revenue == 0).mean():.1%}")

con_datos = peliculas[(peliculas.budget > 0) & (peliculas.revenue > 0)]
print(f"\nPresupuesto medio contando los ceros : USD {peliculas.budget.mean():,.0f}")
print(f"Presupuesto medio sin los ceros       : USD {con_datos.budget.mean():,.0f}")

budget  == 0: 69.7%
revenue == 0: 64.7%

Presupuesto medio contando los ceros : USD 8,766,792
Presupuesto medio sin los ceros       : USD 35,582,138


### 2.4 Sin nota no es lo mismo que mala nota

Cuando `vote_count = 0`, la nota también vale 0. Esos títulos no tienen una
evaluación pésima: **no tienen evaluación**. Incluirlos en un promedio de
satisfacción arrastra la media hacia abajo por una razón que no existe.

In [6]:
sin_votos = series.vote_count == 0
print(f"Series sin ninguna calificación: {sin_votos.sum():,} ({sin_votos.mean():.1%})")
print(f"De ellas, con nota 0: {(series.loc[sin_votos, 'vote_average'] == 0).sum():,}")

# Caso borde: unos pocos registros traen nota sin ningún voto que la respalde.
# Es una nota huérfana y el ETL la descarta junto con el resto de `sin_senal`.
huerfanas = sin_votos & (series.vote_average != 0)
print(f"Notas huérfanas (0 votos pero nota distinta de 0): {huerfanas.sum()}")

print(f"\nNota media contando los ceros : {series.vote_average.mean():.2f}")
print(f"Nota media sólo con votos     : {series.loc[~sin_votos, 'vote_average'].mean():.2f}")
print("\nLa diferencia de 1,6 puntos no refleja ninguna caída de calidad:")
print("es el efecto de tratar 'sin evaluación' como si fuera 'evaluación pésima'.")

Series sin ninguna calificación: 3,674 (23.0%)
De ellas, con nota 0: 3,661
Notas huérfanas (0 votos pero nota distinta de 0): 13

Nota media contando los ceros : 5.42
Nota media sólo con votos     : 7.02

La diferencia de 1,6 puntos no refleja ninguna caída de calidad:
es el efecto de tratar 'sin evaluación' como si fuera 'evaluación pésima'.


### 2.5 La cohorte 2025 todavía no es comparable

Sus títulos no han tenido tiempo de acumular interacciones. Mezclarla con el
resto haría parecer que el catálogo empeoró de golpe.

In [7]:
todo = pd.concat([peliculas.assign(formato="Película"),
                  series.assign(formato="Serie")], ignore_index=True)
por_anio = (todo.assign(sin_votos=todo.vote_count == 0)
            .groupby("release_year").sin_votos.mean() * 100).round(1)
print(por_anio.to_string())

release_year
2010    15.8
2011    14.6
2012    13.5
2013    13.8
2014    11.0
2015    11.8
2016     9.5
2017     8.5
2018     9.0
2019     8.8
2020    10.7
2021     9.8
2022     8.3
2023     7.2
2024     7.8
2025    68.3


## 3. Integración y saneamiento

Las siete decisiones de saneamiento están documentadas en `src/etl.py`. Aquí
sólo se ejecuta el proceso y se revisa el resultado.

In [8]:
catalogo = etl.construir_catalogo()
print(f"Catálogo integrado: {catalogo.shape[0]:,} filas × {catalogo.shape[1]} columnas\n")
etl.auditoria(catalogo)

Catálogo integrado: 31,991 filas × 25 columnas



,control,n,cobertura
0,Registros integrados,31991,100%
1,Películas,16000,50.0%
2,Series,15991,50.0%
3,Títulos sin ninguna interacción,4560,14.3%
4,Títulos sin género declarado,1079,3.4%
5,Títulos sin país informado,2261,7.1%
6,Películas con presupuesto informado,4847,30.3% de las películas
7,Películas con ROI calculable,3540,22.1% de las películas
8,"Cohorte 2025 (parcial, excluida de tendencias)",1991,6.2%


In [9]:
catalogo[["id_titulo", "formato", "titulo", "anio", "idioma_original",
          "segmento", "atencion", "interacciones", "satisfaccion",
          "sin_senal", "roi"]].head(8)

,id_titulo,formato,titulo,anio,idioma_original,segmento,atencion,interacciones,satisfaccion,sin_senal,roi
0,PEL-41371,Película,#1 Cheerleader Camp,2010,en,Película · Inglés,7.203,98,4.800,False,NaN
1,PEL-44115,Película,127 Hours,2010,en,Película · Inglés,37.971,7606,7.100,False,1.983333
2,PEL-44982,Película,13,2010,en,Película · Inglés,12.177,819,5.800,False,NaN
3,PEL-58857,Película,13 Assassins,2010,ja,Película · No inglés,16.249,1141,7.300,False,2.925857
4,PEL-48015,Película,13Hrs,2010,en,Película · Inglés,7.476,69,4.400,False,NaN
5,PEL-34179,Película,14 Blades,2010,zh,Película · No inglés,11.469,196,6.300,False,NaN
6,PEL-40205,Película,16 Wishes,2010,en,Película · Inglés,26.027,1321,6.314,False,NaN
7,PEL-40912,Película,2001 Maniacs: Field of Screams,2010,en,Película · Inglés,6.459,80,3.950,False,NaN


A partir de aquí se trabaja sobre la **cohorte madura** (2010–2024), que es la
base de todas las cifras del informe.

In [10]:
maduras = M.cargar(solo_maduras=True)
kpis = M.kpis(maduras)
for clave, valor in kpis.items():
    print(f"  {clave:<26} {valor:,.2f}" if isinstance(valor, float)
          else f"  {clave:<26} {valor:,}")

  titulos                    30,000
  atencion_top10             47.55
  catalogo_invisible_pct     10.67
  catalogo_invisible_n       3,202
  satisfaccion_mediana       6.70
  satisfaccion_n             26,798
  atencion_mediana           23.05
  gini_atencion              0.58


## 4. Hallazgo 1 · La atención está extremadamente concentrada

El índice de Gini y la curva de Lorenz son la forma estándar de medir
desigualdad en una distribución. Aplicados a la atención muestran que el
tamaño del catálogo no es una medida de salud.

In [11]:
for pct in (1, 5, 10, 20, 50):
    print(f"  El {pct:>2}% de los títulos concentra "
          f"{M.atencion_en_top(maduras, pct):.1f}% de la atención")
print(f"\n  Índice de Gini de la atención: {M.gini(maduras.atencion):.3f}")
print(f"  Índice de Gini de las interacciones: {M.gini(maduras.interacciones):.3f}")

  El  1% de los títulos concentra 17.7% de la atención
  El  5% de los títulos concentra 35.7% de la atención
  El 10% de los títulos concentra 47.5% de la atención
  El 20% de los títulos concentra 62.3% de la atención
  El 50% de los títulos concentra 85.7% de la atención

  Índice de Gini de la atención: 0.577
  Índice de Gini de las interacciones: 0.855


## 5. Hallazgo 2 · El formato es la variable que más explica el rendimiento

Se usa la **mediana** y no el promedio: la distribución de atención tiene cola
larga y unos pocos estrenos masivos desplazarían la media.

In [12]:
con_senal = maduras[~maduras.sin_senal]

resumen_formato = pd.DataFrame({
    "Títulos": maduras.groupby("formato").size(),
    "Atención mediana": maduras.groupby("formato").atencion.median(),
    "Atención promedio": maduras.groupby("formato").atencion.mean(),
    "Nota mediana": con_senal.groupby("formato").satisfaccion.median(),
    "Catálogo invisible %": maduras.groupby("formato").sin_senal.mean() * 100,
}).round(2)
print(resumen_formato.to_string())

multiplo = (resumen_formato.loc["Serie", "Atención mediana"]
            / resumen_formato.loc["Película", "Atención mediana"])
print(f"\nUna serie capta {multiplo:.1f} veces la atención de una película.")
print("Nótese la diferencia entre mediana y promedio: por eso no se usa el promedio.")

          Títulos  Atención mediana  Atención promedio  Nota mediana  Catálogo invisible %
formato                                                                                   
Película    15000             11.05              19.49           6.4                  1.06
Serie       15000             37.39              66.67           7.2                 20.29

Una serie capta 3.4 veces la atención de una película.
Nótese la diferencia entre mediana y promedio: por eso no se usa el promedio.


### El formato que más rinde es también el más riesgoso

Una de cada cinco series no registra ninguna interacción. La recomendación no
puede ser «comprar series» sin más: hay que comprarlas con un filtro de
riesgo.

In [13]:
M.por_segmento(maduras)

,titulos,atencion_mediana,invisible_pct,satisfaccion_mediana
segmento,,,,
Serie · Inglés,4145,40.24,7.86,7.19
Serie · No inglés,10855,36.36,25.03,7.30
Película · Inglés,8985,11.93,0.16,6.26
Película · No inglés,6015,9.96,2.41,6.60


In [14]:
# Dentro de las series, el idioma modula el riesgo más que el techo de rendimiento
solo_series = maduras[maduras.formato == "Serie"]
por_idioma = (solo_series.groupby("idioma_original")
              .agg(titulos=("titulo", "size"),
                   atencion=("atencion", "median"),
                   invisible_pct=("sin_senal", lambda s: s.mean() * 100)))
por_idioma["nota"] = (solo_series[~solo_series.sin_senal]
                      .groupby("idioma_original").satisfaccion.median())
por_idioma.query("titulos >= 300").sort_values("invisible_pct").round(2)

,titulos,atencion,invisible_pct,nota
idioma_original,,,,
tr,332,38.31,6.63,7.22
en,4145,40.24,7.86,7.19
ja,1780,31.23,8.03,7.50
es,870,54.60,11.03,7.10
ko,1298,37.48,11.56,7.19
pt,329,58.50,22.19,7.03
zh,2349,32.30,33.76,7.40
hi,328,58.09,34.76,7.00
fr,548,43.08,40.88,7.00


Los idiomas de la parte superior de la tabla —bajo porcentaje de catálogo
invisible con atención y nota altas— son los candidatos naturales para
concentrar la inversión. Los del final exigen un piloto antes de comprometer
presupuesto.

## 6. Hallazgo 3 · Hay inventario pagado que nadie ve

Cruzar atención y satisfacción sobre sus medianas convierte un conjunto de
30.000 títulos en cuatro decisiones distintas.

In [15]:
puntos, umbral_atencion, umbral_nota = M.cuadrantes(maduras)
print(f"Umbrales (medianas de la cohorte): atención {umbral_atencion:.2f} · "
      f"nota {umbral_nota:.2f}\n")
M.resumen_cuadrantes(maduras)

Umbrales (medianas de la cohorte): atención 21.33 · nota 6.70



,titulos,atencion_mediana,satisfaccion_mediana,pct_series,pct_no_ingles,pct_catalogo
cuadrante,,,,,,
Motores,8761,41.99,7.6,83.77,60.45,32.69
Ruido,4638,38.64,6.0,71.19,56.49,17.31
Joyas ocultas,5088,11.65,7.2,17.37,57.86,18.99
Rezagados,8311,9.73,5.9,5.20,37.88,31.01


In [16]:
# Ejemplos del cuadrante que abre la oportunidad más barata
(puntos.query("cuadrante == 'Joyas ocultas'")
       .nlargest(10, "satisfaccion")
       [["titulo", "formato", "anio", "idioma_original",
         "atencion", "interacciones", "satisfaccion"]])

,titulo,formato,anio,idioma_original,atencion,interacciones,satisfaccion
398,"Inácio Garapa, Um Matuto Sonhador",Película,2010,pt,4.553,1,10.0
1002,10,Serie,2010,fr,15.042,1,10.0
1154,Daam,Serie,2010,ur,11.451,1,10.0
1178,Des crimes presque parfaits,Serie,2010,fr,13.860,1,10.0
1303,Hallo K3,Serie,2010,nl,13.789,1,10.0
1362,Interpol,Serie,2010,fr,14.435,1,10.0
1515,Mori no Asagao,Serie,2010,ja,19.562,2,10.0
1718,Super Trio Game Master,Serie,2010,cn,18.252,1,10.0
1872,UFC Ultimate Insider,Serie,2010,en,14.010,1,10.0
1918,Yariyan,Serie,2010,ur,15.889,1,10.0


## 7. Hallazgo 4 · El dinero está en el tramo equivocado

Sólo una parte de las películas informa presupuesto y recaudación, de modo que
las conclusiones financieras aplican a ese subconjunto y así se declara en el
informe.

In [17]:
cobertura = maduras.roi.notna().sum() / (maduras.formato == "Película").sum()
print(f"Películas con ROI calculable: {maduras.roi.notna().sum():,} "
      f"({cobertura:.1%} de las películas)\n")
M.rentabilidad(maduras)

Películas con ROI calculable: 3,515 (23.4% de las películas)



,peliculas,roi_mediano,pct_bajo_equilibrio,presupuesto_total_musd,atencion_mediana,satisfaccion_mediana
tramo_presupuesto,,,,,,
< 1 M,197,3.14,28.93,98.21,10.79,6.50
1 – 10 M,1196,1.20,45.74,6599.83,12.67,6.54
10 – 50 M,1446,1.48,39.56,36298.37,18.72,6.50
50 – 100 M,343,2.21,23.32,25735.17,34.44,6.60
> 100 M,333,2.77,10.51,56482.87,59.68,6.80


El tramo de 1 a 10 millones de dólares es el que más presupuesto moviliza y el
que peor retorno entrega. Y subir el presupuesto tampoco mejora la nota:

In [18]:
M.correlaciones(maduras)

,budget,revenue,atencion,interacciones,satisfaccion
budget,1.00,0.71,0.56,0.59,0.06
revenue,0.71,1.00,0.62,0.69,0.25
atencion,0.56,0.62,1.00,0.72,0.29
interacciones,0.59,0.69,0.72,1.00,0.27
satisfaccion,0.06,0.25,0.29,0.27,1.00


In [19]:
fin = maduras[maduras.roi.notna()].copy()
fin["decil"] = pd.qcut(fin.budget, 10, labels=range(1, 11))
evolucion = fin.groupby("decil", observed=True).agg(
    presupuesto_mediano=("budget", "median"),
    atencion=("atencion", "median"),
    nota=("satisfaccion", "median"))
indexado = (evolucion[["atencion", "nota"]] / evolucion[["atencion", "nota"]].iloc[0] * 100)
evolucion.join(indexado, rsuffix="_indice").round(1)

,presupuesto_mediano,atencion,nota,atencion_indice,nota_indice
decil,,,,,
1,1000000.0,11.0,6.6,100.0,100.0
2,4000000.0,12.7,6.5,115.4,98.5
3,6100000.0,12.8,6.5,116.6,98.5
4,9000000.0,13.2,6.5,119.5,98.5
5,13000000.0,15.5,6.5,140.9,99.0
6,20000000.0,17.6,6.5,159.4,98.5
7,27360000.0,19.0,6.5,172.8,98.5
8,40000000.0,24.8,6.5,225.1,98.8
9,75000000.0,34.4,6.6,312.6,100.0


Del decil más barato al más caro la atención se multiplica por más de cinco y
la nota se mueve apenas tres puntos de índice. **El presupuesto compra
visibilidad, no agrado.**

Ambas series se indexan a base 100 porque miden en unidades distintas: un
gráfico de doble eje inventaría una correlación que los datos no tienen.

## 8. Generación de las figuras del informe

Las ocho figuras se producen con el mismo módulo que usa el informe, de modo
que lo que se ve aquí es exactamente lo que se entrega.

In [20]:
G.generar_todo()

Generando figuras en /home/daniel/Escritorio/visualizacion/EP1_StreamView_Analytics/images
  ✓ f1_concentracion_atencion.png


  ✓ f2_rendimiento_formato.png
  ✓ f3_catalogo_invisible.png


  ✓ f4_matriz_cuadrantes.png


  ✓ f5_rentabilidad_tramos.png
  ✓ f6_presupuesto_atencion_nota.png


  ✓ f7_internacionalizacion.png


  ✓ f8_generos.png
Listo.


## 9. Síntesis

| Paso | Contenido |
|---|---|
| **Situación** | La inversión en contenido se decide por volumen, no por rendimiento. |
| **Hallazgo** | La atención está concentrada, una parte del catálogo no llega a nadie y el formato serie rinde varias veces más que la película. |
| **Implicancia** | Miles de millones están en el tramo de películas de peor retorno, y más presupuesto no mejora la satisfacción. |
| **Acción** | Reasignar ese tramo a series con filtro de riesgo, activar las joyas ocultas, instalar una puerta de 90 días e instrumentar telemetría propia. |

### Lo que estos datos **no** permiten afirmar

No existe telemetría propia de reproducción: «atención» es un índice externo de
popularidad, no horas vistas, y no hay variables de suscripción, dispositivo ni
churn. El análisis describe el rendimiento del **catálogo**, no el
comportamiento de los **usuarios**.

---

### Declaración de uso de inteligencia artificial

Se utilizó **Claude (Anthropic)** como apoyo en la exploración del conjunto de
datos, la revisión crítica de decisiones de diseño visual, la redacción del
informe y la depuración del código. Todas las cifras se calculan desde el
conjunto de datos original mediante el código de este proyecto y fueron
verificadas por el equipo, que asume la responsabilidad íntegra del contenido.